# 🔍 Surface Defect Detection Pipeline
---

## CELL 1 — Theory & Workflow

### What is Surface Defect Detection?

Surface defect detection means automatically finding problems (like scratches, cracks, or missing parts) in photos of manufactured items — such as steel plates or printed circuit boards (PCBs).  
In factories, inspecting parts manually is slow and error-prone. A computer vision model can scan hundreds of parts per minute with consistent accuracy.

### What is YOLOv8?

**YOLO** stands for *You Only Look Once* — a family of real-time object detection models.  
**YOLOv8** (by Ultralytics) is the latest version. It:
- Is fast enough to run on a laptop CPU.
- Is very accurate even with small datasets.
- Has a simple Python API — just a few lines of code to train and use.
- Works for both images and video.

YOLOv8 draws **bounding boxes** around each defect and labels it with the class name and confidence score.

### Our 4 Defect Classes

| Class ID | Name | Examples |
|----------|------|----------|
| 0 | scratch | surface scratch, scuff, seam |
| 1 | dent | pit, pitted surface, inclusion |
| 2 | crack | crack, fissure, fracture |
| 3 | missing_part | missing part, open circuit, pin-hole |

### Full Workflow

1. **Download datasets** — NEU-DET (metal) + DeepPCB/Kaggle PCB dataset.
2. **Relabel & merge** — Remap class names to our 4 unified classes in YOLO format.
3. **Train YOLOv8** — Train for 10-20 epochs on the merged dataset.
4. **Inference** — Test on images and video to see bounding boxes.
5. **Streamlit app** — Upload images/video or use webcam for real-time detection.

In [1]:
!kaggle datasets download -d akhatova/pcb-defects -p data/pcb/ --unzip


Dataset URL: https://www.kaggle.com/datasets/akhatova/pcb-defects
License(s): unknown




  0%|          | 0.00/1.88G [00:00<?, ?B/s]
  0%|          | 1.00M/1.88G [00:01<58:07, 577kB/s]
  0%|          | 2.00M/1.88G [00:02<31:41, 1.06MB/s]
  0%|          | 3.00M/1.88G [00:02<20:46, 1.61MB/s]
  0%|          | 4.00M/1.88G [00:02<14:58, 2.24MB/s]
  0%|          | 6.00M/1.88G [00:02<09:07, 3.67MB/s]
  0%|          | 8.00M/1.88G [00:03<06:29, 5.14MB/s]
  1%|          | 10.0M/1.88G [00:03<04:39, 7.17MB/s]
  1%|          | 12.0M/1.88G [00:03<03:53, 8.59MB/s]
  1%|          | 15.0M/1.88G [00:03<03:09, 10.5MB/s]
  1%|          | 18.0M/1.88G [00:03<02:26, 13.6MB/s]
  1%|          | 20.0M/1.88G [00:03<02:32, 13.1MB/s]
  1%|          | 22.0M/1.88G [00:03<02:17, 14.4MB/s]
  1%|▏         | 25.0M/1.88G [00:04<01:53, 17.4MB/s]
  1%|▏         | 27.0M/1.88G [00:04<02:02, 16.2MB/s]
  2%|▏         | 30.0M/1.88G [00:04<02:04, 15.9MB/s]
  2%|▏         | 33.0M/1.88G [00:04<02:03, 16.0MB/s]
  2%|▏         | 36.0M/1.88G [00:04<01:47, 18.3MB/s]
  2%|▏         | 38.0M/1.88G [00:04<01:57, 16.8MB/s]
  

In [2]:
import os
from pathlib import Path

# Set the working directory to the project root based on the notebook location
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
print("Working directory is now:", os.getcwd())

# Fix 2: See exactly what's inside data/metal/
print("\n=== Contents of data/metal/ ===")
for root, dirs, files in os.walk('data/metal'):
    level = root.replace('data/metal', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if files:
        print(f'{indent}  → {len(files)} files, e.g: {files[0]}')

Working directory is now: c:\Users\vicky\Desktop\CV-CrackDetector

=== Contents of data/metal/ ===


---
## CELL 2 — Install Packages

In [3]:
# Install all required packages.
# Run this cell once. After installation, you don't need to run it again.

# ultralytics  — The YOLOv8 framework (train, detect, export)
# opencv-python — Image and video processing
# streamlit    — Turns Python scripts into web apps
# pillow       — Image format conversions (used by streamlit)
# pandas       — Create and export CSV reports

%pip install ultralytics opencv-python streamlit pillow pandas --quiet

print('✅ All packages installed!')

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed!


---
## CELL 3 — Download Datasets

We use two datasets:
1. **NEU-DET** — Steel surface defects (6 original classes).
2. **DeepPCB** — PCB defects (6 original classes).

Choose **one of the methods** below based on what you have access to.

In [4]:
import os
import zipfile
import shutil

# ── Create base folder structure ──────────────────────────
os.makedirs('data/metal', exist_ok=True)   # For NEU-DET images + labels
os.makedirs('data/pcb',   exist_ok=True)   # For PCB images + labels
os.makedirs('data/merged/images/train', exist_ok=True)
os.makedirs('data/merged/images/val',   exist_ok=True)
os.makedirs('data/merged/images/test',  exist_ok=True)
os.makedirs('data/merged/labels/train', exist_ok=True)
os.makedirs('data/merged/labels/val',   exist_ok=True)
os.makedirs('data/merged/labels/test',  exist_ok=True)
os.makedirs('models', exist_ok=True)

print('✅ Directory structure created.')

# ════════════════════════════════════════════════════════
# METHOD 1: Using Kaggle API (recommended)
# ════════════════════════════════════════════════════════
# Step 1: Install kaggle package
# !pip install kaggle --quiet

# Step 2: Place your kaggle.json API key at ~/.kaggle/kaggle.json
#   Download it from: https://www.kaggle.com/settings → API → Create New Token

# Step 3: Download NEU-DET (metal defects)
 #!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database -p data/metal/ --unzip

# Step 4: Download PCB defect dataset
 #!kaggle datasets download -d akhatova/pcb-defects -p data/pcb/ --unzip

# ════════════════════════════════════════════════════════
# METHOD 2: Clone DeepPCB from GitHub (free, no account needed)
# ════════════════════════════════════════════════════════
# !git clone https://github.com/tangsanli5201/DeepPCB.git data/pcb/DeepPCB

# ════════════════════════════════════════════════════════
# METHOD 3: Manual download
# ════════════════════════════════════════════════════════
# 1. Go to https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database
# 2. Click Download, unzip, and place contents inside:  data/metal/
#
# 1. Go to https://www.kaggle.com/datasets/akhatova/pcb-defects
# 2. Click Download, unzip, and place contents inside:  data/pcb/

print('📁 After downloading, make sure images are in data/metal/ and data/pcb/')
print('Expected structures:')
print('  data/metal/  → subfolders per class OR mixed images+labels')
print('  data/pcb/    → images + annotation files')

✅ Directory structure created.
📁 After downloading, make sure images are in data/metal/ and data/pcb/
Expected structures:
  data/metal/  → subfolders per class OR mixed images+labels
  data/pcb/    → images + annotation files


---
## CELL 4 — Data Processing & Label Mapping

We remap each dataset's original class names to our 4 unified classes:

| Original Label | → | Our Class |
|---|---|---|
| scratch, crazing, scuff, seam, rolled-in-scale | → | scratch (0) |
| inclusion, pit, pitted_surface, copper | → | dent (1) |
| crack, fissure, fracture, mousebite | → | crack (2) |
| missing_part, open, short, spur, pin-hole, broken_edge | → | missing_part (3) |

In [6]:
import os
import glob
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path

# Set the working directory to the project root based on the notebook location
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

# ── Class mapping: NEU-DET class name → our unified class ID ──
CLASS_MAP = {
    'crazing':         0,  # → scratch
    'scratch':         0,
    'scratches':       0,
    'patches':         0,  # → scratch
    'rolled-in-scale': 0,  # → scratch
    'rolled-in_scale': 0,  # → scratch (underscore version)
    'inclusion':       1,  # → dent
    'pitted_surface':  1,  # → dent
    'crack':           2,  # → crack
    'missing_part':    3,  # → missing_part
}


def convert_voc_xml_to_yolo(xml_path: str, img_w: int, img_h: int) -> list:
    """Convert one Pascal VOC XML file to YOLO format label lines."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    label_lines = []
    for obj in root.findall('object'):
        class_name   = obj.find('name').text.strip().lower().replace(' ', '_')
        new_class_id = CLASS_MAP.get(class_name, -1)

        if new_class_id == -1:
            print(f"    ⚠ Unknown class '{class_name}' — skipping")
            continue

        bndbox = obj.find('bndbox')
        xmin   = float(bndbox.find('xmin').text)
        ymin   = float(bndbox.find('ymin').text)
        xmax   = float(bndbox.find('xmax').text)
        ymax   = float(bndbox.find('ymax').text)

        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h

        label_lines.append(f"{new_class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    return label_lines


def find_image_in_subfolders(img_dir: str, stem: str) -> str:
    """
    Search for an image file by stem name inside any subfolder.
    e.g. looks for 'crazing_1.jpg' inside images/crazing/, images/scratches/, etc.
    Returns the full path if found, else None.
    """
    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        # Search recursively in all subfolders
        matches = glob.glob(os.path.join(img_dir, '**', stem + ext), recursive=True)
        if matches:
            return matches[0]  # Return first match
    return None


def process_split(split_name: str, metal_dir: str, out_img_dir: str, out_lbl_dir: str):
    """
    Process one split (train or validation).
    Handles images stored in class subfolders inside images/.
    """
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    anno_dir = os.path.join(metal_dir, split_name, 'annotations')
    img_dir  = os.path.join(metal_dir, split_name, 'images')

    if not os.path.isdir(anno_dir):
        print(f"  ⚠ Not found: {anno_dir}")
        return 0

    xml_files = glob.glob(os.path.join(anno_dir, '*.xml'))
    print(f"\n  [{split_name}] Found {len(xml_files)} XML annotation files")

    copied  = 0
    skipped = 0

    for xml_path in xml_files:
        stem = Path(xml_path).stem  # e.g. "crazing_1"

        # Find the image inside any subfolder
        img_path = find_image_in_subfolders(img_dir, stem)

        if img_path is None:
            skipped += 1
            continue

        # Get image dimensions from XML
        tree     = ET.parse(xml_path)
        root     = tree.getroot()
        size_tag = root.find('size')
        img_w    = int(size_tag.find('width').text)
        img_h    = int(size_tag.find('height').text)

        # Convert XML → YOLO labels
        label_lines = convert_voc_xml_to_yolo(xml_path, img_w, img_h)

        if not label_lines:
            skipped += 1
            continue

        # Copy image to merged output folder (flat, no subfolders)
        dst_img = os.path.join(out_img_dir, stem + Path(img_path).suffix)
        shutil.copy2(img_path, dst_img)

        # Save YOLO .txt label
        txt_path = os.path.join(out_lbl_dir, stem + '.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(label_lines))

        copied += 1

    print(f"    ✅ Saved:   {copied} image+label pairs")
    print(f"    ⏭ Skipped: {skipped} (no matching image or no valid labels)")
    return copied


# ── Process both splits ───────────────────────────────────
metal_dir  = 'data/metal'
merged_dir = 'data/merged'

print("=== Processing NEU-DET Metal Dataset ===")

process_split('train',      metal_dir,
              os.path.join(merged_dir, 'images', 'train'),
              os.path.join(merged_dir, 'labels', 'train'))

process_split('validation', metal_dir,
              os.path.join(merged_dir, 'images', 'val'),
              os.path.join(merged_dir, 'labels', 'val'))

# ── Create test split from 10% of val ────────────────────
print("\n=== Creating test split (10% of val) ===")

val_images = glob.glob(os.path.join(merged_dir, 'images', 'val', '*'))
random.seed(42)
random.shuffle(val_images)
test_images = val_images[:max(1, len(val_images) // 10)]

os.makedirs(os.path.join(merged_dir, 'images', 'test'), exist_ok=True)
os.makedirs(os.path.join(merged_dir, 'labels', 'test'), exist_ok=True)

for img_path in test_images:
    stem = Path(img_path).stem
    shutil.move(img_path, os.path.join(merged_dir, 'images', 'test', Path(img_path).name))
    lbl_src = os.path.join(merged_dir, 'labels', 'val', stem + '.txt')
    if os.path.exists(lbl_src):
        shutil.move(lbl_src, os.path.join(merged_dir, 'labels', 'test', stem + '.txt'))

print(f"  Moved {len(test_images)} images → test")

# ── Final summary ─────────────────────────────────────────
print("\n=== Final Dataset Summary ===")
for split in ['train', 'val', 'test']:
    n_img = len(glob.glob(os.path.join(merged_dir, 'images', split, '*')))
    n_lbl = len(glob.glob(os.path.join(merged_dir, 'labels', split, '*')))
    print(f"  {split:6s}: {n_img} images, {n_lbl} labels")

print("\n✅ Done! Ready for Cell 5 (training).")


Working directory: C:\Users\vicky\Desktop\CV-CrackDetector
=== Processing NEU-DET Metal Dataset ===
  ⚠ Not found: data/metal\train\annotations
  ⚠ Not found: data/metal\validation\annotations

=== Creating test split (10% of val) ===
  Moved 0 images → test

=== Final Dataset Summary ===
  train : 0 images, 0 labels
  val   : 0 images, 0 labels
  test  : 0 images, 0 labels

✅ Done! Ready for Cell 5 (training).


In [1]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

# ── PCB class name → our unified class ID ──
PCB_CLASS_MAP = {
    'missing_hole':     3,  # → missing_part
    'mouse_bite':       2,  # → crack
    'open_circuit':     3,  # → missing_part
    'short':            3,  # → missing_part
    'spur':             1,  # → dent
    'spurious_copper':  1,  # → dent
}

def convert_pcb_xml_to_yolo(xml_path: str, img_w: int, img_h: int) -> list:
    """Convert one PCB Pascal VOC XML file to YOLO format label lines."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    label_lines = []
    for obj in root.findall('object'):
        # Get class name: lowercase, replace spaces with underscore
        class_name   = obj.find('name').text.strip().lower().replace(' ', '_')
        new_class_id = PCB_CLASS_MAP.get(class_name, -1)

        if new_class_id == -1:
            print(f"    ⚠ Unknown PCB class '{class_name}' — skipping")
            continue

        bndbox = obj.find('bndbox')
        xmin   = float(bndbox.find('xmin').text)
        ymin   = float(bndbox.find('ymin').text)
        xmax   = float(bndbox.find('xmax').text)
        ymax   = float(bndbox.find('ymax').text)

        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h

        label_lines.append(f"{new_class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    return label_lines


def process_pcb_dataset(pcb_dir: str, merged_dir: str, train_ratio=0.7, val_ratio=0.2):
    """
    Process the Kaggle PCB dataset and add it to data/merged/.
    Images and XMLs are organized in class subfolders.
    Splits into train/val/test automatically.
    """
    anno_base = os.path.join(pcb_dir, 'Annotations')
    img_base  = os.path.join(pcb_dir, 'images')

    if not os.path.isdir(anno_base) or not os.path.isdir(img_base):
        print(f"PCB dataset not found at: {pcb_dir}")
        print("Expected folders: Annotations/ and images/")
        return 0

    # Collect all (image_path, xml_path) pairs across all class folders
    all_pairs = []

    for class_folder in os.listdir(anno_base):
        # Skip rotation folders — they are augmented duplicates
        if 'rotation' in class_folder.lower():
            continue

        anno_folder = os.path.join(anno_base, class_folder)
        img_folder  = os.path.join(img_base,  class_folder)

        if not os.path.isdir(anno_folder):
            continue

        xml_files = glob.glob(os.path.join(anno_folder, '*.xml'))
        print(f"  {class_folder}: {len(xml_files)} files")

        for xml_path in xml_files:
            stem = Path(xml_path).stem  # e.g. "01_missing_hole_01"

            # Find matching image
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                candidate = os.path.join(img_folder, stem + ext)
                if os.path.exists(candidate):
                    img_path = candidate
                    break

            if img_path:
                all_pairs.append((img_path, xml_path))

    print(f"\n  Total PCB pairs found: {len(all_pairs)}")

    # Shuffle and split into train/val/test
    import random
    random.seed(99)
    random.shuffle(all_pairs)

    n       = len(all_pairs)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    splits = {
        'train': all_pairs[:n_train],
        'val':   all_pairs[n_train : n_train + n_val],
        'test':  all_pairs[n_train + n_val:],
    }

    saved   = 0
    skipped = 0

    for split_name, pairs in splits.items():
        out_img_dir = os.path.join(merged_dir, 'images', split_name)
        out_lbl_dir = os.path.join(merged_dir, 'labels', split_name)
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_lbl_dir, exist_ok=True)

        for img_path, xml_path in pairs:
            stem = Path(xml_path).stem

            # Get image size from XML
            tree     = ET.parse(xml_path)
            root     = tree.getroot()
            size_tag = root.find('size')

            # Some XMLs may not have size tag — get from image instead
            if size_tag is not None:
                img_w = int(size_tag.find('width').text)
                img_h = int(size_tag.find('height').text)
            else:
                # Read image dimensions using PIL as fallback
                from PIL import Image as PILImage
                with PILImage.open(img_path) as im:
                    img_w, img_h = im.size

            # Convert to YOLO labels
            label_lines = convert_pcb_xml_to_yolo(xml_path, img_w, img_h)

            if not label_lines:
                skipped += 1
                continue

            # Copy image (add 'pcb_' prefix to avoid name clashes with metal images)
            dst_img = os.path.join(out_img_dir, 'pcb_' + stem + Path(img_path).suffix)
            shutil.copy2(img_path, dst_img)

            # Save label
            dst_lbl = os.path.join(out_lbl_dir, 'pcb_' + stem + '.txt')
            with open(dst_lbl, 'w') as f:
                f.write('\n'.join(label_lines))

            saved += 1

    print(f"\n  ✅ Saved:   {saved} PCB image+label pairs")
    print(f"  ⏭ Skipped: {skipped}")
    return saved


# ── Run PCB processing ────────────────────────────────────
print("=== Processing Kaggle PCB Dataset ===\n")
def find_pcb_dataset_dir(base_dirs=('data/pcb', 'notebooks/data/pcb')):
    """Find a PCB dataset folder containing both Annotations/ and images/."""
    for base_dir in base_dirs:
        base_path = Path(base_dir)
        if not base_path.exists():
            continue

        candidates = [base_path] + [p for p in base_path.iterdir() if p.is_dir()]
        for candidate in candidates:
            if (candidate / 'Annotations').is_dir() and (candidate / 'images').is_dir():
                return str(candidate)
    return None

pcb_dataset_dir = find_pcb_dataset_dir()
if pcb_dataset_dir:
    print(f"Using PCB dataset folder: {pcb_dataset_dir}")
    process_pcb_dataset(pcb_dataset_dir, 'data/merged')
else:
    print("No PCB dataset found under data/pcb. Skipping PCB merge.")

# ── Updated final summary (metal + PCB combined) ─────────
print("\n=== Combined Dataset Summary (Metal + PCB) ===")
for split in ['train', 'val', 'test']:
    n_img = len(glob.glob(os.path.join('data/merged/images', split, '*')))
    n_lbl = len(glob.glob(os.path.join('data/merged/labels', split, '*')))
    print(f"  {split:6s}: {n_img} images, {n_lbl} labels")

print("\n✅ Both datasets merged! Ready for Cell 5 (training).")


=== Processing Kaggle PCB Dataset ===

No PCB dataset found under data/pcb. Skipping PCB merge.

=== Combined Dataset Summary (Metal + PCB) ===
  train : 0 images, 0 labels
  val   : 0 images, 0 labels
  test  : 0 images, 0 labels

✅ Both datasets merged! Ready for Cell 5 (training).


---
## CELL 5 — Train YOLOv8

We create a YAML config file that tells YOLOv8 where the data is and what classes to detect.  
Then we call `model.train()` — that's all it takes!

**Estimated training time:**  
- CPU only: ~5–10 minutes per epoch → 10 epochs ≈ 1–2 hours  
- GPU (CUDA): ~30 seconds per epoch → 10 epochs ≈ 5 minutes

In [2]:
import os
from pathlib import Path
import yaml
from ultralytics import YOLO

# Set the working directory to the project root based on the notebook location
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

# ── Step 1: Create the dataset YAML config ────────────────
merged_dir = os.path.abspath('data/merged')  # Full absolute path (YOLOv8 prefers this)

dataset_config = {
    'path':  merged_dir,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    4,
    'names': ['scratch', 'dent', 'crack', 'missing_part']
}

yaml_path = 'data/merged/defects.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f'✅ YAML config saved to: {yaml_path}')
print()

# ── Step 2: Load YOLOv8 nano model ────────────────────────
model = YOLO('yolov8n.pt')
print('✅ YOLOv8n model loaded.')
print('   Starting training — this takes ~60-90 mins on CPU ☕\n')

# ── Step 3: Train ─────────────────────────────────────────
results = model.train(
    data=yaml_path,
    epochs=20,         # Changed from 10 → 20 for better accuracy
    imgsz=640,
    batch=8,           # Reduce to 4 if you get a memory error
    name='defect_v1',
    patience=5,
    device='cpu',
    workers=0,         # Keep at 0 on Windows — avoids multiprocessing errors
    verbose=True,
)

print('\n✅ Training complete!')
print('Best model saved at:', results.save_dir)

Working directory: C:\Users\vicky\Desktop\CV-CrackDetector
✅ YAML config saved to: data/merged/defects.yaml

✅ YOLOv8n model loaded.
   Starting training — this takes ~60-90 mins on CPU ☕

New https://pypi.org/project/ultralytics/8.4.31 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.13.5 torch-2.9.0+cpu CPU (11th Gen Intel Core i5-11320H @ 3.20GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/merged/defects.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.

FileNotFoundError: [34m[1mtrain: [0mError loading data from C:\Users\vicky\Desktop\CV-CrackDetector\data\merged\images\train
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

---
## CELL 6 — Inference on a Single Image

In [ ]:
import cv2
import os
import glob
from ultralytics import YOLO

# ── Load the trained model ────────────────────────────────
# After training, copy best.pt to models/ (done in Cell 8).
# Here we first try the trained model, then fall back to the test set.

trained_model_path = 'models/yolov8n_defects.pt'
default_model_path = 'yolov8n.pt'  # Pretrained, detects COCO classes — just for demo

if os.path.exists(trained_model_path):
    model = YOLO(trained_model_path)  # Your trained defect detector
    print(f'✅ Loaded trained model from: {trained_model_path}')
else:
    model = YOLO(default_model_path)  # Fallback: not trained on defects
    print(f'⚠️  Trained model not found. Using default YOLOv8n (COCO classes).')
    print('   Run Cell 5 first, then Cell 8 to save your trained model.')

# ── Find a test image ─────────────────────────────────────
# Look for any image in the test set (or any image you provide)
test_images = glob.glob('data/merged/images/test/*.jpg') + \
              glob.glob('data/merged/images/test/*.png') + \
              glob.glob('data/merged/images/test/*.bmp')

if test_images:
    test_image_path = test_images[0]   # Use the first test image found
    print(f'Using test image: {test_image_path}')
else:
    # If no test set yet, use any image you have
    test_image_path = 'your_test_image.jpg'  # ← Change this to your image path
    print('No test images found. Please set test_image_path manually.')

# ── Run inference ─────────────────────────────────────────
if os.path.exists(test_image_path):
    # Run the model on the image
    results = model(test_image_path, conf=0.25)  # conf=0.25 means show detections ≥ 25% confidence

    # Get the annotated image as a numpy array (BGR format)
    # .plot() draws bounding boxes and labels on the image
    annotated_image = results[0].plot()

    # Print detection summary
    print(f'\nDetections found: {len(results[0].boxes)}')
    for box in results[0].boxes:
        class_id   = int(box.cls[0])
        confidence = float(box.conf[0])
        coords     = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
        print(f'  Class {class_id} | Confidence: {confidence:.2f} | Box: {[round(c) for c in coords]}')

    # ── Display with OpenCV ───────────────────────────────
    # Resize for display if the image is very large
    display_img = cv2.resize(annotated_image, (800, 600)) if annotated_image.shape[1] > 800 else annotated_image

    cv2.imshow('Defect Detection Result', display_img)  # Open a window
    print('\nPress any key to close the image window...')
    cv2.waitKey(0)       # Wait until user presses a key
    cv2.destroyAllWindows()  # Close the window

    # ── Also save the result ──────────────────────────────
    os.makedirs('reports', exist_ok=True)
    output_path = 'reports/result_image.jpg'
    cv2.imwrite(output_path, annotated_image)
    print(f'\n✅ Annotated image saved to: {output_path}')
else:
    print(f'Image not found: {test_image_path}')

---
## CELL 7 — Inference on a Video

In [ ]:
import cv2
import os
from ultralytics import YOLO

# ── Configuration ─────────────────────────────────────────
VIDEO_PATH  = 'your_test_video.mp4'   # ← Change this to your video file path
OUTPUT_PATH = 'reports/result_video.mp4'  # Where to save the annotated output video
CONF_THRESHOLD = 0.25                 # Minimum confidence to show a detection
PROCESS_EVERY_N_FRAMES = 2            # Process every 2nd frame (faster on CPU)

# ── Load model ────────────────────────────────────────────
trained_model_path = 'models/yolov8n_defects.pt'
if os.path.exists(trained_model_path):
    model = YOLO(trained_model_path)
    print(f'✅ Loaded trained model.')
else:
    model = YOLO('yolov8n.pt')
    print('⚠️ Using default YOLOv8n (not trained on defects).')

# ── Open the video file ───────────────────────────────────
if not os.path.exists(VIDEO_PATH):
    print(f'Video not found: {VIDEO_PATH}')
    print('Please set VIDEO_PATH to a valid .mp4 or .avi file.')
else:
    cap = cv2.VideoCapture(VIDEO_PATH)  # Open the video file

    # Get video properties
    fps          = cap.get(cv2.CAP_PROP_FPS)             # Frames per second
    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f'Video: {frame_width}x{frame_height} @ {fps:.1f} FPS, {total_frames} frames')

    # ── Set up video writer ───────────────────────────────
    # This saves the annotated video to a file
    os.makedirs('reports', exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 output
    writer = cv2.VideoWriter(
        OUTPUT_PATH, fourcc, fps,
        (frame_width, frame_height)
    )

    frame_idx    = 0
    total_defects = 0

    print('Processing video... Press Q in the display window to stop early.')

    # ── Process frame by frame ────────────────────────────
    while cap.isOpened():
        ret, frame = cap.read()   # Read next frame
        if not ret:
            break  # End of video

        frame_idx += 1

        # Run inference only every N frames (to save time on CPU)
        if frame_idx % PROCESS_EVERY_N_FRAMES == 0:
            results        = model(frame, conf=CONF_THRESHOLD, verbose=False)
            annotated      = results[0].plot()  # Draw boxes on the frame
            total_defects += len(results[0].boxes)
        else:
            annotated = frame  # Use the original frame for skipped frames

        # Write the annotated frame to the output video
        writer.write(annotated)

        # Show the annotated frame in a window (optional — remove if too slow)
        display = cv2.resize(annotated, (800, 450))  # Resize for display
        cv2.putText(display, f'Frame {frame_idx}/{total_frames}',
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow('Video Defect Detection', display)

        # Press 'q' to quit early
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print('\n⏹ Stopped early by user.')
            break

    # Release resources
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    print(f'\n✅ Done! Total defections detected: {total_defects}')
    print(f'   Annotated video saved to: {OUTPUT_PATH}')

---
## CELL 8 — Save the Trained Model

After training, YOLOv8 saves the best model to `runs/detect/defect_v1/weights/best.pt`.  
We copy it to `models/yolov8n_defects.pt` so the Streamlit app can find it.

In [ ]:
import os
import shutil
import glob
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

# Find the newest trained YOLO weights inside this project
candidates = glob.glob(str(PROJECT_ROOT / 'runs' / 'detect' / '**' / 'weights' / 'best.pt'), recursive=True)

if candidates:
    best_pt_path = max(candidates, key=os.path.getmtime)
    os.makedirs('models', exist_ok=True)
    destination = 'models/yolov8n_defects.pt'
    shutil.copy2(best_pt_path, destination)

    size_mb = os.path.getsize(destination) / (1024 * 1024)
    print(f'✅ Model copied to: {destination}  ({size_mb:.1f} MB)')
    print(f'   Source: {best_pt_path}')
    print('\nYou can now:')
    print('  1. Run Cell 6 to test on a single image')
    print('  2. Run Cell 7 to test on a video')
    print('  3. Launch the Streamlit app:')
    print('     streamlit run streamlit_app/app.py')
else:
    print(f'❌ Could not find best.pt under: {PROJECT_ROOT / "runs" / "detect"}')

In [ ]:
import glob
test_images = glob.glob('data/merged/images/test/*')
print(f'Test images available: {len(test_images)}')
print('First 3:', test_images[:3])

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

model = YOLO('models/yolov8n_defects.pt')

# Try with very low confidence to see if anything is detected at all
results = model('data/merged/images/test/crazing_275.jpg', conf=0.01, verbose=True)

print(f'\nDetections at conf=0.01: {len(results[0].boxes)}')
for box in results[0].boxes:
    print(f'  Class {int(box.cls[0])} | Conf: {float(box.conf[0]):.3f}')